# Grad-CAM + FGSM Adversarial Attack on ONE image

See **visually** what happens to a model's "attention" before and after a tiny adversarial attack.

- Pretrained **ResNet50** (knows 1000 ImageNet classes)
- **FGSM** attack on a single image
- **Grad-CAM** heatmaps before & after, plus the noise itself

> Note: a single image needs almost no compute, so 2×T4 is overkill. This code uses **one GPU** on purpose — multi-GPU for one image would only add complexity and slowdown. It still runs fine on your 2×T4 instance.


In [ ]:
# ============================================================
# CELL 1 — Setup
# ============================================================
import torch, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt
from torchvision import models, transforms
from PIL import Image
import urllib.request, json

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)   # single GPU is plenty for one image

In [ ]:
# ============================================================
# CELL 2 — Load a pretrained model (already knows 1000 classes)
# ============================================================
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.eval().to(device)

# ImageNet class names (for readable predictions)
url = "https://raw.githubusercontent.com/raghakot/keras-vis/master/resources/imagenet_class_index.json"
idx2label = {int(k): v[1] for k, v in json.load(urllib.request.urlopen(url)).items()}

In [ ]:
# ============================================================
# CELL 3 — Load ONE image
# Option A: upload your own and set the path below
# Option B: download a sample (needs Internet ON in Kaggle settings)
# ============================================================
IMG_PATH = "dog.jpg"
try:
    img = Image.open(IMG_PATH).convert("RGB")
except FileNotFoundError:
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg", IMG_PATH)
    img = Image.open(IMG_PATH).convert("RGB")

# We keep the image in [0,1] pixel space, and normalize INSIDE the model call.
# This makes the adversarial noise easy to see in real pixels later.
to_tensor = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])
x = to_tensor(img).unsqueeze(0).to(device)   # shape [1,3,224,224], values 0..1

mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
normalize = lambda t: (t - mean) / std       # convert 0..1 -> model's expected input

In [ ]:
# ============================================================
# CELL 4 — Grad-CAM helper
# Idea: look at the LAST conv layer's feature maps, weight them
# by how much each map influences the chosen class, then overlay.
# ============================================================
class GradCAM:
    def __init__(self, model, layer):
        self.model = model
        self.acts = self.grads = None
        layer.register_forward_hook(lambda m, i, o: setattr(self, "acts", o.detach()))
        layer.register_full_backward_hook(lambda m, gi, go: setattr(self, "grads", go[0].detach()))

    def __call__(self, inp, class_idx=None):
        out = self.model(normalize(inp))           # forward pass
        if class_idx is None:
            class_idx = out.argmax(1).item()        # default: top predicted class
        self.model.zero_grad()
        out[0, class_idx].backward()                # gradients flow to last conv layer

        weights = self.grads.mean(dim=(2, 3), keepdim=True)   # importance of each map
        cam = F.relu((weights * self.acts).sum(1, keepdim=True))  # weighted sum, keep positives
        cam = F.interpolate(cam, size=(224, 224), mode="bilinear", align_corners=False)
        cam = cam.squeeze().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)  # normalize 0..1
        return cam, class_idx, out.softmax(1)[0, class_idx].item()

cam_tool = GradCAM(model, model.layer4[-1])   # last conv block of ResNet50

In [ ]:
# ============================================================
# CELL 5 — FGSM attack (Fast Gradient Sign Method)
# Push every pixel a tiny step in the direction that INCREASES
# the loss for the true class -> model gets fooled.
# ============================================================
def fgsm_attack(inp, true_class, epsilon=0.02):
    inp = inp.clone().detach().requires_grad_(True)
    out = model(normalize(inp))
    loss = F.cross_entropy(out, torch.tensor([true_class], device=device))
    model.zero_grad(); loss.backward()
    adv = inp + epsilon * inp.grad.sign()   # step along the gradient's sign
    return torch.clamp(adv, 0, 1).detach()  # keep valid pixel range

In [ ]:
# ============================================================
# CELL 6 — Run everything: clean -> attack -> compare
# ============================================================
# 1) Grad-CAM on the CLEAN image
cam_clean, clean_cls, clean_conf = cam_tool(x)

# 2) Build the adversarial image (attack the model's own top guess)
x_adv = fgsm_attack(x, clean_cls, epsilon=0.02)

# 3) Grad-CAM on the ADVERSARIAL image
cam_adv, adv_cls, adv_conf = cam_tool(x_adv)

# 4) The noise itself = difference between the two images
noise = (x_adv - x).squeeze().cpu().permute(1, 2, 0).numpy()

print(f"BEFORE attack -> {idx2label[clean_cls]} ({clean_conf:.1%})")
print(f"AFTER  attack -> {idx2label[adv_cls]} ({adv_conf:.1%})")

In [ ]:
# ============================================================
# CELL 7 — Visualize the whole story in one figure
# ============================================================
def show(ax, img_np, title, cam=None):
    ax.imshow(img_np)
    if cam is not None:
        ax.imshow(cam, cmap="jet", alpha=0.5)   # heatmap overlay
    ax.set_title(title, fontsize=10); ax.axis("off")

orig = x.squeeze().cpu().permute(1, 2, 0).numpy()
advn = x_adv.squeeze().cpu().permute(1, 2, 0).numpy()

fig, ax = plt.subplots(2, 3, figsize=(13, 8))
show(ax[0, 0], orig, f"1) Original\n{idx2label[clean_cls]} {clean_conf:.0%}")
show(ax[0, 1], (noise - noise.min()) / (noise.max() - noise.min() + 1e-8),
     "2) The added NOISE\n(amplified to be visible)")
show(ax[0, 2], advn, f"3) Adversarial\n{idx2label[adv_cls]} {adv_conf:.0%}")
show(ax[1, 0], orig, "4) Grad-CAM BEFORE", cam_clean)
show(ax[1, 1], orig, "5) Original vs adv\n(look identical to us)")
show(ax[1, 2], advn, "6) Grad-CAM AFTER", cam_adv)
plt.tight_layout(); plt.show()

## What you should see

- Panels 1 & 3 look **identical** to your eyes, yet the prediction **flips**.
- Panel 2 shows the noise blown up so it's even visible.
- Panel 6's heatmap **shifts away** from the real object — that shift *is* the attack at work.

### Things to try
- Raise `epsilon` (e.g. `0.05`) → easier to fool, more visible noise.
- Lower it (`0.005`) → see how little perturbation is needed.
- Swap FGSM for an iterative **PGD** attack for a stronger, more surgical effect.
